In [1]:
import os
import torch
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
from tqdm import tqdm
import torch.optim as optim
import torch.nn.functional as F
import torch.nn as nn

import pandas as pd
import numpy as np

final_df = pd.read_csv("/kaggle/input/lmsys-chatbot-arena/test.csv")
train = pd.read_csv("/kaggle/input/lmsys-chatbot-arena/train.csv")
train["kfold"] = 0
final_df.head()



,id,prompt,response_a,response_b
0,3297560222,"[""What can you tell me about Maarten van Vulpe...","[""Maarten van Vulpen (also spelled Marten or M...","[""Maarten van Vulpen is not a widely known pub..."
1,2556155375,"[""is cebu island a good place to travel to in ...","[""Yes, Cebu Island is a great place to visit i...","[""Cebu Island can be a good place to travel to..."
2,1793939629,"[""Hi, we've been trying to reach you about you...","[""I'm glad you reached out, but I must let you...","[""\""I understand that you're calling about my ..."
3,2562993561,"[""write the complete code implementation ONLY ...","[""Sure, here is the complete code implementati...","[""Sure! Here is an example implementation of S..."
4,2111345399,"[""\u00bfFinancializing Poverty. Labor and Risk...","[""Financializing poverty refers to the process...","[""\""Financializing Poverty: Labor and Risk in ..."


In [2]:
final_df["text"] = (
    "User prompt: "
    + final_df["prompt"]
    + "\n\nModel A :\n"
    + final_df["response_a"]
    + "\n\n--------\n\nModel B:\n"
    + final_df["response_b"]
)
print(final_df["text"][0])
train["text"] = (
    "User prompt: "
    + train["prompt"]
    + "\n\nModel A :\n"
    + train["response_a"]
    + "\n\n--------\n\nModel B:\n"
    + train["response_b"]
)



User prompt: ["What can you tell me about Maarten van Vulpen?","What do you know about Fonto?"]

Model A :
["Maarten van Vulpen (also spelled Marten or Martin) was a Dutch Golden Age painter, mainly known for his still life paintings, particularly those featuring flowers. He was born in Utrecht in 1608 and is believed to have studied under the renowned painter Jan van Kessel the Elder, who himself was a student of Jan Brueghel the Elder. Van Vulpen became a member of the Utrecht Guild of Saint Luke in 1631 and achieved the position of dean in 1648.\n\nHis still life paintings often depicted luxurious arrangements of flowers, fruits, and other natural elements, as well as animals such as butterflies and birds. Van Vulpen's paintings often display an impressive attention to detail and a fine sense of color composition. While he was heavily influenced by Flemish still-life painters, his works demonstrate a unique style of his own.\n\nDuring his career, Maarten van Vulpen produced numerous

In [3]:
print(len(final_df))
final_df.head()



5748


,id,prompt,response_a,response_b,text
0,3297560222,"[""What can you tell me about Maarten van Vulpe...","[""Maarten van Vulpen (also spelled Marten or M...","[""Maarten van Vulpen is not a widely known pub...","User prompt: [""What can you tell me about Maar..."
1,2556155375,"[""is cebu island a good place to travel to in ...","[""Yes, Cebu Island is a great place to visit i...","[""Cebu Island can be a good place to travel to...","User prompt: [""is cebu island a good place to ..."
2,1793939629,"[""Hi, we've been trying to reach you about you...","[""I'm glad you reached out, but I must let you...","[""\""I understand that you're calling about my ...","User prompt: [""Hi, we've been trying to reach ..."
3,2562993561,"[""write the complete code implementation ONLY ...","[""Sure, here is the complete code implementati...","[""Sure! Here is an example implementation of S...","User prompt: [""write the complete code impleme..."
4,2111345399,"[""\u00bfFinancializing Poverty. Labor and Risk...","[""Financializing poverty refers to the process...","[""\""Financializing Poverty: Labor and Risk in ...","User prompt: [""\u00bfFinancializing Poverty. L..."


In [4]:
final_texts = final_df["text"].values
batch_size = 8
num_classes = 3
kfolds = 1  # only one fold needed for the pretrained models


class GRUClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim=256,
        hidden_dim=128,
        hidden_dim2=64,
        num_classes=num_classes,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, hidden_dim2)
        self.fc2 = nn.Linear(hidden_dim2, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        output, h_n = self.gru(x)
        output = self.fc(h_n[-1])  # use final hidden state
        logits = self.fc2(output)
        return logits


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)


def encode(sentence):
    return torch.tensor([vocab.get(w, 1) for w in sentence])


def predict(text, model):
    tokens = text.split()
    encoded = encode(tokens).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(encoded)
        probs = F.softmax(logits, dim=1)
        return probs


final_df["winner_model_a"] = 0.0
final_df["winner_model_b"] = 0.0
final_df["winner_tie"] = 0.0

epoches_to_use = [0]  # only one epoch entry for the single fold

for kfold in range(kfolds):
    print(f"Prediction fold: {kfold}")

    test_texts = train[train["kfold"] == kfold]["text"].values
    train_texts = train[train["kfold"] != kfold]["text"].values

    test_tokenized = [t.split() for t in test_texts]
    train_tokenized = [t.split() for t in train_texts]

    vocab = {"<pad>": 0, "<unk>": 1}
    for word in Counter(w for sent in test_tokenized for w in sent):
        vocab[word] = len(vocab)
    for word in Counter(w for sent in train_tokenized for w in sent):
        vocab[word] = len(vocab)

    model_path = f"/kaggle/input/gru-train-5fold/gru_classifier_kfold_{kfold}_epoch_{epoches_to_use[kfold]}.pth"
    if os.path.isfile(model_path):
        model_loaded = GRUClassifier(vocab_size=len(vocab)).to(device)
        model_loaded.load_state_dict(torch.load(model_path, map_location=device))
        model_loaded.eval()
    else:
        # Fallback dummy model that produces uniform probabilities
        class DummyModel(nn.Module):
            def forward(self, x):
                batch = x.size(0)
                return torch.zeros(batch, num_classes, device=device)

        model_loaded = DummyModel().to(device)
        model_loaded.eval()
        print("Warning: pretrained model not found, using uniform dummy predictions.")

    for i, text in enumerate(final_texts):
        probs = predict(text, model_loaded)
        final_df.at[i, "winner_model_a"] += float(probs[0][0]) / kfolds
        final_df.at[i, "winner_model_b"] += float(probs[0][1]) / kfolds
        final_df.at[i, "winner_tie"] += float(probs[0][2]) / kfolds



Using: cuda
Prediction fold: 0


In [5]:
prob_sum = final_df[["winner_model_a", "winner_model_b", "winner_tie"]].sum(axis=1)
final_df["winner_model_a"] /= prob_sum
final_df["winner_model_b"] /= prob_sum
final_df["winner_tie"] /= prob_sum

final_df[["id", "winner_model_a", "winner_model_b", "winner_tie"]].to_csv(
    "submission.csv", index=False
)
print("Done")

Done
